# Figure: Transmission Maps

Shows a grid of recovered transmission maps next to the input processed images and the
re-processed predictions.

**Columns:** Target (input image) | Recovered TXM | Prediction (forward of TXM)  
**Requires:** `nb_run.ipynb` to have been executed.

Output: `build/transmission_maps.png`

In [ ]:
%%capture
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    DRIVE_ROOT = "/content/drive/MyDrive/thesis"  # <-- edit if your Drive path differs

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", f"{DRIVE_ROOT}/notebooks/requirements.txt",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "-e", DRIVE_ROOT,
    ])

    # Change to figures/ so load_results() can find the cached data
    os.chdir(f"{DRIVE_ROOT}/figures")


In [ ]:
import sys, os, glob, joblib

import matplotlib.pyplot as plt
from chest_xray_sim.visualization import plot_results

def load_results(results_dir="results"):
    hits = sorted(glob.glob(f"{results_dir}/*/imgs.joblib"))
    if not hits:
        raise FileNotFoundError(f"No results in '{results_dir}'. Run nb_run.ipynb first.")
    d = os.path.dirname(hits[-1])
    print(f"Loading from {d}")
    return {
        "imgs":       joblib.load(f"{d}/imgs.joblib"),
        "masks":      joblib.load(f"{d}/masks.joblib"),
        "txm":        joblib.load(f"{d}/txm.joblib"),
        "pred":       joblib.load(f"{d}/pred.joblib"),
        "weights":    joblib.load(f"{d}/weights.joblib"),
        "metrics":    joblib.load(f"{d}/metrics.joblib"),
        "priors":     joblib.load(f"{d}/priors.joblib"),
        "seg_labels": joblib.load(f"{d}/seg_labels.joblib"),
    }

In [ ]:
r = load_results()
imgs, txm, pred, masks = r["imgs"], r["txm"], r["pred"], r["masks"]
print(f"{imgs.shape[0]} images, resolution {imgs.shape[1]}×{imgs.shape[2]}")

In [ ]:
N_SHOW = 4
fig, ax = plot_results(imgs, txm, pred, segmentations=masks, max_images=N_SHOW)
fig.savefig("build/transmission_maps.png", bbox_inches="tight")
plt.show()